In [1]:
import wandb
import pandas as pd
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, ttest_ind

from utils import compute_percent_improvement, WandbParser


api = wandb.Api()

In [2]:
def standard_post_processing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"model_type": "model_name", "data_name": "dataset"})
    df["dataset"] = df["dataset"].str.replace("_", " ")
    df["model_name"] = df["model_name"].str.replace("_", " ")
    df["metric"] = df["metric"].str.replace("test/", "")

    # Corrected logic
    # bsarec_filter = (
    #     (df["model_name"] == "BSARec") &
    #     (
    #         ((df["dataset"] == "MIND") & (df["c"] == 7) & (df["lr"] == 0.005) & (df["num_attention_heads"] == 1) & (df["alpha"] == 0.5)) |
    #         ((df["dataset"] == "taobao small") & (df["c"] == 9) & (df["lr"] == 0.005) & (df["num_attention_heads"] == 4) & (df["alpha"] == 0.7))
    #     )
    # )
    
    # # Invert to keep only these runs for BSARec; all others (non-matching BSARec configs) are dropped
    # df = df[~((df["model_name"] == "BSARec") & ~bsarec_filter)]

    # df = df.drop(columns=["c", "lr", "num_attention_heads", "alpha"])

    return df

In [3]:
wandb_parser = WandbParser(entity="bsarec", api=api, verbose=True)

# models = ["SASRec", "BERT4Rec", "FMLPRec", "duoRec", "FeaRec", "BSARec", "BSARec_Wavelet"]
models = ["BSARec", "FMLPRec", "duoRec", "FeaRec", "BERT4Rec"]
datasets = [
    'taobao_small',
    'MIND'
    # 'Sports_and_Outdoors',
    # 'Toys_and_Games',
    # 'Yelp'
]
metrics = ["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"]
# seed = [37, 5166, 3817]
seed = None

# add reproduction runs
wandb_parser.register_project(
    project="new_datasets",
    model_type=models,
    data_name=datasets,
    seed=seed
)

# # add extra runs
# wandb_parser.register_project(
#     project="BSARec_Wavelet_Tuning_Results",
#     model_type=models[-1:],
#     data_name=datasets,
#     seed=seed
# )

Added new_datasets to parser


In [4]:
df = wandb_parser.parse(
    cfg=["model_type", "data_name", "seed", "c", "lr", "num_attention_heads", "alpha"],
    summary=[f"test/{metric_name}" for metric_name in metrics],
    post_processing=standard_post_processing
)
df_immutable = df.copy(deep=True)

df.head(5)

Loading runs from new_datasets
Applying post processing


,model_name,dataset,seed,c,lr,num_attention_heads,alpha,metric,value
0,BSARec,MIND,42,5.0,0.0005,1,0.7,HR@5,0.084092
1,BSARec,MIND,42,5.0,0.0005,1,0.7,HR@10,0.129352
2,BSARec,MIND,42,5.0,0.0005,1,0.7,HR@20,0.192473
3,BSARec,MIND,42,5.0,0.0005,1,0.7,NDCG@5,0.056693
4,BSARec,MIND,42,5.0,0.0005,1,0.7,NDCG@10,0.071207


In [ ]:
# amount of entries per model
print("Amount of entries per model:")
print(df["model_name"].value_counts())
# on

bsarec_mind  = (
    (df["model_name"] == "BSARec")
    & (df["dataset"] == "MIND")
    & (df["c"] == 7)
    & (df["lr"] == 0.005)
    & (df["num_attention_heads"] == 1)
    & (df["alpha"] == 0.5)
)
bsarec_taobao = (
    (df["model_name"] == "BSARec")
    & (df["dataset"] == "taobao_small")
    & (df["c"] == 9)
    & (df["lr"] == 0.005)
    & (df["num_attention_heads"] == 4)
    & (df["alpha"] == 0.7)
)

# keep everything that is NOT BSARec, plus only those two BSARec runs
keep = (df["model_name"] != "BSARec") | bsarec_mind | bsarec_taobao

df = df[keep]
print("Amount of entries per model after removing BSARec runs:")
print(df["model_name"].value_counts())

df_immutable = df.copy(deep=True)

Amount of entries per model:
model_name
BSARec      1824
BERT4Rec      42
FMLPRec       42
duoRec        18
Name: count, dtype: int64
Amount of entries per model after removing BSARec runs:
model_name
BSARec      84
BERT4Rec    42
FMLPRec     42
duoRec      18
Name: count, dtype: int64


In [6]:
# verifu number of runs
df_immutable.loc[:, ["model_name", "dataset", "seed"]].groupby(["model_name", "dataset"]).nunique()

seed
model_name dataset           
BERT4Rec   MIND             3
           taobao small     3
BSARec     MIND             3
FMLPRec    MIND             4
           taobao small     3
duoRec     MIND             3

# Setting

In [7]:
DEBUG = False
PAIRED = False
CUT_OFF = True

ttest_fn = ttest_rel if PAIRED else ttest_ind

# Paired T-test

In [8]:
df_ttest = df_immutable.copy(deep=True)
df_seed = df_immutable.copy(deep=True)
df_ttest = df_ttest.drop(columns=["seed"])
mean_df = df_ttest.groupby(['model_name', 'dataset', 'metric'])['value'].mean().reset_index()
pivot_df = mean_df.pivot_table(index=['dataset', 'metric'], columns='model_name', values='value')

assert len(pivot_df.columns) >= 2

results = []

target_model = "BSARec"
signficant_col = "significant"

for (dataset, metric), row in pivot_df.iterrows():
    row_wo_target = row.drop(labels=target_model, errors='ignore').sort_values(ascending=False)

    if row_wo_target.count() < 1:
        continue

    best_other_model = row_wo_target.index[0]

    # Extract values for both models
    target_vals = df_seed[(df_seed['model_name'] == target_model) &
                           (df_seed['dataset'] == dataset) &
                           (df_seed['metric'] == metric)]
    target_vals = target_vals.sort_values(["seed"])
    target_seeds = target_vals["seed"].values



    best_vals = df_seed[(df_seed['model_name'] == best_other_model) &
                         (df_seed['dataset'] == dataset) &
                         (df_seed['metric'] == metric)]
    best_vals = best_vals.sort_values(["seed"])
    best_seeds = best_vals["seed"].values

    target_vals = target_vals["value"].values
    best_vals = best_vals["value"].values

    # Ensure seeds match for paired t-test
    if PAIRED and not np.all(target_seeds == best_seeds):
        print(f"Skipping {dataset} - {metric}: Seed mismatch between {target_model} and {best_other_model}")
        print(f"Seeds: [{target_model}] {target_seeds} vs [{best_other_model}] {best_seeds}")
        continue
    elif not PAIRED and len(target_seeds) != len(best_seeds):
        if not CUT_OFF:
            print(f"Skipping {dataset} - {metric}: Length mismatch between {target_model} and {best_other_model}")
            print(f"Length: [{target_model}] {len(target_seeds)} vs [{best_other_model}] {len(best_seeds)}")
            continue
        max_samples = min(len(target_seeds), len(best_seeds))
        print(f"Taking first {max_samples} samples {dataset} - {metric}: Length mismatch between {target_model} and {best_other_model}")
        print(f"Length: [{target_model}] {len(target_seeds)} vs [{best_other_model}] {len(best_seeds)}")
        target_vals = target_vals[:max_samples]
        best_vals = best_vals[:max_samples]


    t_stat, p_value = ttest_fn(target_vals, best_vals)

    if DEBUG:
        results.append({
            'dataset': dataset,
            'metric': metric,
            'target_model': target_model,
            'best_other_model': best_other_model,
            'target_mean': target_vals.mean(),
            'best_other_mean': best_vals.mean(),
            signficant_col: p_value < 0.05,
        })
    else:
        results.append({
            'dataset': dataset,
            'metric': metric,
            signficant_col: p_value < 0.05,
        })

stats_df = pd.DataFrame(results)

if len(stats_df) > 0:
    stats_df = stats_df.set_index(["dataset", "metric"])
    display(stats_df.head())

Taking first 4 samples MIND - HR@10: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 14 vs [BERT4Rec] 4
Taking first 4 samples MIND - HR@20: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 14 vs [BERT4Rec] 4
Taking first 4 samples MIND - HR@5: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 14 vs [BERT4Rec] 4
Taking first 4 samples MIND - NDCG@10: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 14 vs [BERT4Rec] 4
Taking first 4 samples MIND - NDCG@20: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 14 vs [BERT4Rec] 4
Taking first 4 samples MIND - NDCG@5: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 14 vs [BERT4Rec] 4
Taking first 0 samples taobao small - HR@10: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 0 vs [BERT4Rec] 3
Taking first 0 samples taobao small - HR@20: Length mismatch between BSARec and BERT4Rec
Length: [BSARec] 0 vs [BERT4Rec] 3
Taking first 0 samples taobao small - HR@5: Le

significant
dataset metric              
MIND    HR@10           True
        HR@20           True
        HR@5           False
        NDCG@10        False
        NDCG@20         True

# Main Table

In [9]:
df = df_immutable.copy(deep=True).drop(columns=["seed"])
df = (
    df.groupby(["model_name", "dataset", "metric"])
      .agg(["mean"])
      .stack(future_stack=True)
      .reset_index()
      .pivot_table(
          index=['dataset', 'metric'],
          columns='model_name',
          values='value'
      )
)

# create Improv. column with NaN for all rows first
rel_improvement_col= "Diff."
df[rel_improvement_col] = np.nan

print(df.columns, df.index)
# compute improvement only on 'mean' rows
df.loc[:, rel_improvement_col] = df.apply(compute_percent_improvement, axis=1)


Index(['BERT4Rec', 'BSARec', 'FMLPRec', 'duoRec', 'Diff.'], dtype='object', name='model_name') MultiIndex([(        'MIND',   'HR@10'),
            (        'MIND',   'HR@20'),
            (        'MIND',    'HR@5'),
            (        'MIND', 'NDCG@10'),
            (        'MIND', 'NDCG@20'),
            (        'MIND',  'NDCG@5'),
            ('taobao small',   'HR@10'),
            ('taobao small',   'HR@20'),
            ('taobao small',    'HR@5'),
            ('taobao small', 'NDCG@10'),
            ('taobao small', 'NDCG@20'),
            ('taobao small',  'NDCG@5')],
           names=['dataset', 'metric'])


KeyError: 'BSARec'

In [ ]:
# add statistic significance column
df = df.join(stats_df, how="left")

In [ ]:
df.head()

BERT4Rec    BSARec   FMLPRec    duoRec     Diff.  significant
dataset metric                                                                
MIND    HR@10    0.173941  0.167204  0.098738  0.169181 -3.872877         True
        HR@20    0.250644  0.241153  0.152853  0.242860 -3.787019         True
        HR@5     0.114287  0.110970  0.062427  0.112126 -2.902147        False
        NDCG@10  0.094748  0.091995  0.052886  0.092611 -2.905064        False
        NDCG@20  0.114079  0.110614  0.066481  0.111183 -3.037900         True

In [ ]:
# sort metrics
df = df.copy().reset_index()
df["metric"] = pd.Categorical(df["metric"], categories=metrics, ordered=True)

df = df.sort_values(["dataset", "metric"])
df = df.set_index(["dataset", "metric"])

# re-order columns to match the models list
print(f"cols: {df.columns}")
col_order = [mname.replace("_", " ") for mname in models if mname.replace("_", " ") in df.columns] + [rel_improvement_col, signficant_col]
df = df[col_order]

cols: Index(['BERT4Rec', 'BSARec', 'FMLPRec', 'duoRec', 'Diff.', 'significant'], dtype='object')


In [ ]:
# drop
# df = df.dropna()

In [ ]:
df.index

MultiIndex([(        'MIND',    'HR@5'),
            (        'MIND',   'HR@10'),
            (        'MIND',   'HR@20'),
            (        'MIND',  'NDCG@5'),
            (        'MIND', 'NDCG@10'),
            (        'MIND', 'NDCG@20'),
            ('taobao small',    'HR@5'),
            ('taobao small',   'HR@10'),
            ('taobao small',   'HR@20'),
            ('taobao small',  'NDCG@5'),
            ('taobao small', 'NDCG@10'),
            ('taobao small', 'NDCG@20')],
           names=['dataset', 'metric'])

In [ ]:
def format_row(row: pd.Series, rel_improvement_col, significant_col, show_second_best=True) -> list:
    formatted_row = {}

    stat_cols = [rel_improvement_col, significant_col]

    row = pd.to_numeric(row, errors='coerce')
    top2 = row.drop(stat_cols).nlargest(2).index.tolist()
    first, second = top2

    for name, val in row.items():
        if name in stat_cols:
            continue

        fval = val

        if np.isnan(fval):
            fval = ""
        elif isinstance(val, float):
            fval = f"{val:.4f}"
        elif isinstance(val, str):
            pass
        else:
            fval = str(val)

        if name == first:
            fval = "\\textbf{" + fval + "}"
        elif name == second and show_second_best:
            fval = "\\underline{" + fval + "}"

        formatted_row[name] = fval

    modifier = {
        True: "\\textsuperscript{*}",
        False: ""
    }

    formatted_row[rel_improvement_col] = f"{row[rel_improvement_col]:.2f}" + modifier.get(row[significant_col], " X")

    return pd.Series(formatted_row)

df = df.apply(lambda row: format_row(row, rel_improvement_col, signficant_col, show_second_best=False), axis=1)

In [ ]:
df.index

MultiIndex([(        'MIND',    'HR@5'),
            (        'MIND',   'HR@10'),
            (        'MIND',   'HR@20'),
            (        'MIND',  'NDCG@5'),
            (        'MIND', 'NDCG@10'),
            (        'MIND', 'NDCG@20'),
            ('taobao small',    'HR@5'),
            ('taobao small',   'HR@10'),
            ('taobao small',   'HR@20'),
            ('taobao small',  'NDCG@5'),
            ('taobao small', 'NDCG@10'),
            ('taobao small', 'NDCG@20')],
           names=['dataset', 'metric'])

In [ ]:


# create base latex string
# WARNING: hardcoded column count
latex = df.to_latex(escape=False, na_rep="", multicolumn=True, multirow=True)
latex = latex.replace("\\begin{tabular}{lllllllll}", "\\begin{tabular}{llccccccc}\n\\toprule")
latex = latex.replace("\\\\\n", " \\\\\n\\midrule\n", 1)  # Add \midrule after header
latex = latex.replace("\\cline{1-" + str(len(models) + 3) + "}\n\\bottomrule", "\\bottomrule")
latex = latex.replace("\\cline{1-" + str(len(models) + 3) + "}", "\\midrule")

# fix the header line
lines = latex.splitlines()
toprule_idx = None
for i, line in enumerate(lines):
    if r"\toprule" in line:
        toprule_idx = i
        break

# Replace next two lines after \toprule with single header line
# Assuming they exist
header_line = r"\textbf{Dataset} & \textbf{Metric} &" + " & ".join([f"\\textbf{{{col}}}" for col in col_order if col != signficant_col]) + r" \\"
lines[toprule_idx + 1] = header_line

del lines[toprule_idx+2]

# insert \midrule after new header line if not already present
if r"\midrule" not in lines[toprule_idx+2]:
    lines.insert(toprule_idx+2, r"\midrule")

latex_fixed = "\n".join(lines)
latex_fixed = latex_fixed.replace(f"\\midrule\ndataset & metric &{'  &' * (len(models) - 1)}  &  \\\\\n", "")
latex_fixed = latex_fixed.replace("\\multirow[t]{6}{*}{", "\\multirow{6}{*}{\\centering ")
# latex_fixed = latex_fixed.replace("\\toprule", "", 1)


latex_fixed = r"""
\begin{table*}[h]
    \centering
    \caption{\hc{TODO: avergaed over X runs, """ + ("paired " if PAIRED else "") + r"""\(t\)-test, \(p <.05\) scores marked with \textsuperscript{*}, diff \(\%\) is BSaRec compared to best baseline }}
    \begin{adjustbox}{max width=\textwidth}
    \label{tab:main}
""" + latex_fixed
# latex_fixed = "\\begin{table*}[h]\n\\centering\n" + latex_fixed
# latex_fixed += "\\caption{" + r"averaged over 10 runs, paired \(t\)-test, \(p <.05\) scores marked with \textsuperscript{*}, diff \(\%\) is BSaRec compared to best baseline" + "}\\label{tab:reproduction-std-" + dataset + "}\n"
latex_fixed += "\n\\end{adjustbox}\n\\end{table*}\n"
print(latex_fixed)


\begin{table*}[h]
    \centering
    \caption{\hc{TODO: avergaed over X runs, \(t\)-test, \(p <.05\) scores marked with \textsuperscript{*}, diff \(\%\) is BSaRec compared to best baseline }}
    \begin{adjustbox}{max width=\textwidth}
    \label{tab:main}
\begin{tabular}{lllllll}
\toprule
\textbf{Dataset} & \textbf{Metric} &\textbf{BSARec} & \textbf{FMLPRec} & \textbf{duoRec} & \textbf{BERT4Rec} & \textbf{Diff.} \\
\midrule
dataset & metric &  &  &  &  &  \\
\midrule
\multirow{6}{*}{\centering MIND} & HR@5 & 0.1110 & 0.0624 & 0.1121 & \textbf{0.1143} & -2.90 \\
 & HR@10 & 0.1672 & 0.0987 & 0.1692 & \textbf{0.1739} & -3.87\textsuperscript{*} \\
 & HR@20 & 0.2412 & 0.1529 & 0.2429 & \textbf{0.2506} & -3.79\textsuperscript{*} \\
 & NDCG@5 & 0.0739 & 0.0412 & 0.0743 & \textbf{0.0755} & -2.17 \\
 & NDCG@10 & 0.0920 & 0.0529 & 0.0926 & \textbf{0.0947} & -2.91 \\
 & NDCG@20 & 0.1106 & 0.0665 & 0.1112 & \textbf{0.1141} & -3.04\textsuperscript{*} \\
\cline{1-7}
\multirow{6}{*}{\centering taob

# Standard deviations (per dataset)

In [ ]:
df = df_immutable.copy(deep=True).drop(columns=["seed"])

In [ ]:
df.groupby(["model_name", "dataset"]).nunique()

c  lr  num_attention_heads  alpha  metric  value
model_name dataset                                                       
BERT4Rec   MIND          0   1                    1      0       6     18
           taobao small  0   1                    1      0       6     18
BSARec     MIND          1   1                    1      1       6     24
           taobao small  1   1                    1      1       6      6
FMLPRec    MIND          0   1                    1      0       6     24
           taobao small  0   1                    1      0       6     15
duoRec     MIND          0   1                    1      0       6     18

In [ ]:
datasets = df["dataset"].unique()

def format_std_row(row):
    formatted_row = {
        "model_name": row[("model_name", "")],
        "metric": row[("metric", "")],
        "value": f"${row[('value', 'mean')]:.4f} \pm {row[('value', 'std')]:.4f}$"
    }

    return pd.Series(formatted_row)

for dataset in datasets:
    df_subset = df[df["dataset"] == dataset].drop(columns=["dataset"])
    df_subset = (
        df_subset.groupby(["model_name", "metric"])
        .agg(["mean", "std"])
        .reset_index()
        .apply(format_std_row, axis=1)
        .pivot(index="model_name", columns="metric", values="value")
    )

    # sort columns and index
    df_subset = df_subset.reindex(["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"], axis=1)
    df_subset = df_subset.reindex([n.replace("_", " ") for n in models], axis=0)



    # add begin{table} and end{table} tags
    latex = df_subset.to_latex(escape=False, na_rep="", multicolumn=True, multirow=True)
    latex = "\\begin{table*}[h]\n\\centering\n" + latex
    latex += "\\caption{" + f"Standard deviations on the {dataset} dataset. Averaged over 10 runs." + "}\\label{tab:reproduction-std-" + dataset + "}\n"
    latex += "\\end{table*}\n"
    latex = latex.replace("model_name &  &  &  &  &  &  \\\\\n", "")
    print(latex)
    print()


\begin{table*}[h]
\centering
\begin{tabular}{lllllll}
\toprule
metric & HR@5 & HR@10 & HR@20 & NDCG@5 & NDCG@10 & NDCG@20 \\
\midrule
BSARec & $0.1110 \pm 0.0013$ & $0.1672 \pm 0.0019$ & $0.2412 \pm 0.0019$ & $0.0739 \pm 0.0008$ & $0.0920 \pm 0.0010$ & $0.1106 \pm 0.0010$ \\
FMLPRec & $0.0624 \pm 0.0036$ & $0.0987 \pm 0.0039$ & $0.1529 \pm 0.0081$ & $0.0412 \pm 0.0038$ & $0.0529 \pm 0.0029$ & $0.0665 \pm 0.0028$ \\
duoRec & $0.1121 \pm 0.0024$ & $0.1692 \pm 0.0029$ & $0.2429 \pm 0.0019$ & $0.0743 \pm 0.0016$ & $0.0926 \pm 0.0018$ & $0.1112 \pm 0.0015$ \\
FeaRec &  &  &  &  &  &  \\
BERT4Rec & $0.1143 \pm 0.0016$ & $0.1739 \pm 0.0020$ & $0.2506 \pm 0.0033$ & $0.0755 \pm 0.0009$ & $0.0947 \pm 0.0010$ & $0.1141 \pm 0.0014$ \\
\bottomrule
\end{tabular}
\caption{Standard deviations on the MIND dataset. Averaged over 10 runs.}\label{tab:reproduction-std-MIND}
\end{table*}


\begin{table*}[h]
\centering
\begin{tabular}{lllllll}
\toprule
metric & HR@5 & HR@10 & HR@20 & NDCG@5 & NDCG@10 & NDCG@